In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
import json
from pathlib import Path

from ipysigma import Sigma

import sys
import re
sys.path.append('../')
from source import Align
from textwrap import wrap

In [ ]:
BlastPairWiseAlignmentPivoted = pd.read_csv("../data/processed/BlastPairWiseAlignmentPivoted.cov95.maxseq5.csv", sep="\t")
mask = np.triu(np.ones_like(BlastPairWiseAlignmentPivoted.set_index("Unnamed: 0"), dtype=bool)) 
sns.heatmap(
    BlastPairWiseAlignmentPivoted.set_index("Unnamed: 0"),
    cmap="Blues",
    annot=True,
    fmt=".0f",
    linewidth=.5,
    mask=mask,
)
plt.title("Databases Redudancy (>95% identity)")

In [ ]:
BlastPairWiseAlignmentPivoted

In [ ]:
ClustersCdHit = pd.read_csv(
    "../data/processed/cdhitclusters.defaultsettings.csv",
    sep = "\t",
)
sns.lineplot(
    ClustersCdHit.set_index("Unnamed: 0"),
    marker = "o"
)
plt.xlabel("Identity Threshold (%)")
plt.ylabel("Database Size")

In [ ]:
ClustersCdHit

In [ ]:
PairWiseAlignment = pd.read_csv(
    "../data/filtered/AllDatabases.Paiwise.cov95.maxseq5.tsv", 
    sep = "\t",
    skipinitialspace=True, 
    header=None,
    names = "qseqid sseqid pident length qlen slen qstart qend sstart send evalue bitscore ppos full_qseq full_sseq".split(" ")
)
PairWiseAlignment["qcov"] = np.round((PairWiseAlignment["qend"] - PairWiseAlignment["qstart"] + 1) / (PairWiseAlignment["qlen"]) * 100, 2)
PairWiseAlignment["scov"] = np.round((PairWiseAlignment["send"] - PairWiseAlignment["sstart"] + 1) / (PairWiseAlignment["slen"]) * 100, 2)
PairWiseAlignment["qseqtag"] = PairWiseAlignment["qseqid"].str.split("|").str[1]
PairWiseAlignment["sseqtag"] = PairWiseAlignment["sseqid"].str.split("|").str[1]
PairWiseAlignment["qseqid"] = PairWiseAlignment["qseqid"].str.split("|").str[0]
PairWiseAlignment["sseqid"] = PairWiseAlignment["sseqid"].str.split("|").str[0]

In [ ]:
PairWiseAlignment

In [ ]:
sns.scatterplot(
    data=PairWiseAlignment.loc[PairWiseAlignment["bitscore"] > 70], 
    x="qcov", 
    y="pident",
    size = "bitscore",
    )

In [ ]:
sns.scatterplot(
    data=PairWiseAlignment.loc[PairWiseAlignment["bitscore"] > 80], 
    x="pident", 
    y="ppos",
    size = "bitscore",
    )

In [ ]:
PairWiseAlignment

In [ ]:
with open("../data/processed/MetaDict.cov95.maxseq5.json", "r") as json_file  :
    MetaDict = json.load(json_file)

In [ ]:
SequenceSimilarityGraph = nx.from_pandas_edgelist(
    PairWiseAlignment.loc[PairWiseAlignment["pident"] > 95], 
    source="qseqid", 
    target="sseqid", 
    edge_attr=["pident", "evalue","bitscore", "ppos"]
)
nx.set_node_attributes(SequenceSimilarityGraph, MetaDict)
ConnectedComponents = list(nx.connected_components(SequenceSimilarityGraph))
len(ConnectedComponents)

In [ ]:
ProblematicComponents = set()
ComponentIndex = dict()
DivergenteAnnotatedClasses = []
for i, component in enumerate(ConnectedComponents):
    ComponentGraph = SequenceSimilarityGraph.subgraph(component)
    Classes = [ComponentGraph.nodes[n].get("Drug Class") for n in component]
    if len(set(Classes)) > 1:
        ComponentIndex[i] = {"status": "problematic", "component": component, "classes": set(Classes), "len": len(component)}
        ProblematicComponents = ProblematicComponents.union(component)
        DivergenteAnnotatedClasses.append(set(Classes))
    else:
        ComponentIndex[i] = {"status": "ok", "component": component, "classes": set(Classes), "len": len(component)}

In [ ]:
ClassCount = []
for item in pd.Series(DivergenteAnnotatedClasses):
    for c in item:
        ClassCount.append(c)
pd.Series(ClassCount).value_counts().head(30).to_frame().plot(kind = "barh", figsize=(1,5.5))

In [ ]:
TargetClass = "multidrug"
pd.Series([c for c in DivergenteAnnotatedClasses if TargetClass in c]).value_counts().head(10).to_frame().plot(kind = "barh", figsize=(1,5))

In [ ]:
TargetClass = "multidrug"
ProblematicCompWithTargetClass = set()
for key, value in ComponentIndex.items():
    if value["status"] == "problematic":
        if TargetClass in value["classes"]:
            ProblematicCompWithTargetClass = ProblematicCompWithTargetClass.union(value["component"])
ProblematicComponentsByClass = SequenceSimilarityGraph.subgraph(ProblematicCompWithTargetClass)
meu_layout = {
    "scalingRatio": 50.0,           # Aumente para afastar os grupos
    "gravity": 0.2,                 # Reduza para não amontoar no centro
    "repulsion": 2,               # Aumente para afastar os nós
    # "outboundAttractionDistribution": True, # Empurra hubs para fora
    # "barnesHutOptimize": True,      # Essencial para seus 70k nós
    # "linLogMode": True              # Melhora a definição de clusters biológicos 
}
Sigma(
    ProblematicComponentsByClass, 
    node_size  = ProblematicComponentsByClass.degree(), 
    node_color =  [ProblematicComponentsByClass.nodes[n].get("Drug Class") for n in ProblematicComponentsByClass.nodes()],
    # node_metrics={"community": "louvain"},

    default_edge_type = "curve",
    layout_settings=meu_layout,
    start_layout=30,
    )

In [ ]:
#Select the component index for the specified node bait
SelectedCompIndex = None
NodeBait = "CARD_2015"
for key,value in ComponentIndex.items():
    if NodeBait in value["component"]:
        print("Selected component index:", key)
        SelectedCompIndex = key
        break

if SelectedCompIndex is None:
        raise ValueError(f"{NodeBait} not found in any component index.")
SelectedCluster = SelectedCompIndex

In [ ]:
StringFasta = []
for protein in ComponentIndex[SelectedCluster]["component"]:
    print(f">{protein}|{ProblematicComponentsByClass.nodes[protein]['Drug Class']}\n{ProblematicComponentsByClass.nodes[protein]['Sequence']}")
    StringFasta.append(f">{protein}|{ProblematicComponentsByClass.nodes[protein]['Drug Class']}\n{ProblematicComponentsByClass.nodes[protein]['Sequence']}")

In [ ]:
# Create output directory if it doesn't exist
outdir = Path("../data/processed/selected_components")
outdir.mkdir(parents=True, exist_ok=True)

out_fasta = outdir / f"component_{SelectedCluster}_{NodeBait}.fasta"

def clean_fasta_field(value):
     """
     Clean a value for use in a FASTA header by replacing whitespace and pipe characters with underscores.
     If the value is None, return "NA".
     """
     if value is None:
          return "NA"
     value = str(value).strip()
     value = re.sub(r'\s+', '_', value)  # Replace whitespace with underscores
     value = value.replace("|", "_")  # Replace pipe characters with underscores
     return value

with open(out_fasta, "w") as fasta_file:
     """ Write the sequences of the selected cluster to a FASTA file, including the drug class in the header.
     """
     for protein in sorted(ComponentIndex[SelectedCluster]["component"]):
          drug_class = SequenceSimilarityGraph.nodes[protein].get("Drug Class", "NA")
          sequence = SequenceSimilarityGraph.nodes[protein].get("Sequence", "NA")

          drug_class_clean = clean_fasta_field(drug_class)

          fasta_file.write(f">{protein}|{drug_class_clean}\n")

          for line in wrap(sequence, width=80):
               fasta_file.write(line + "\n")

print (f"Fasta file for component {SelectedCluster} containing {NodeBait} has been saved to: {out_fasta}")

In [ ]:
ProblematicComponentsByClass

In [ ]:
ProblematicCompoLenDist = [value["len"] for key, value in ComponentIndex.items() if value["status"] == "problematic"]

In [ ]:
sns.violinplot(ProblematicCompoLenDist)
plt.xlabel("Component Size")
plt.ylabel("Frequency")
plt.title("Distribution of Problematic Component Sizes")

In [ ]:
ProblematicCompoClassDist = [len(value["classes"]) for key, value in ComponentIndex.items() if value["status"] == "problematic"]

In [ ]:
sns.violinplot(ProblematicCompoClassDist)
plt.xlabel("Number of Classes in Component")
plt.ylabel("Frequency")
plt.title("Distribution of Number of Classes in Problematic Component Sizes")

In [ ]:
AlignedComponent = Align.ProteinAligner(StringFasta)

In [ ]:
def ConcateComponentsBy(CompDict, OriginalGraph, Minsize = None, MinClassNumber = None):
    if Minsize is None and MinClassNumber is None:
        raise ValueError("At least one of Minsize or MinClassNumber must be provided. Both cannot be None.")
    elif Minsize is not None and MinClassNumber is not None:
        raise ValueError("Only one of Minsize or MinClassNumber can be provided. Both cannot be set at the same time.")
    
    ConcateComponents = set()
    for key, value in CompDict.items():
        if Minsize is not None and value["len"] >= Minsize and value["status"] == "problematic":
            ConcateComponents = ConcateComponents.union(value["component"])

        elif MinClassNumber is not None and len(value["classes"]) >= MinClassNumber and value["status"] == "problematic":
            ConcateComponents = ConcateComponents.union(value["component"])            
    return OriginalGraph.subgraph(ConcateComponents),ConcateComponents